# 01 · Auditoría de datos y leakage — TweetAge_2026_PUJ

**Fase 0 del plan.** Objetivo: caracterizar los datos reales, verificar la integridad de la
descarga, documentar el esquema, cuantificar el ruido de etiqueta y **medir la exposición a
leakage por duplicados**, que es el riesgo dominante de este corpus: no existe identificador
de autor, por lo que no es posible agrupar por usuario y el control debe hacerse por
clústeres de near-duplicados (decisión D-003).

**Reglas respetadas.** El archivo de test no tiene etiquetas y no se usa para ajustar nada:
los clústeres se construyen sobre texto (sin etiquetas) y toda métrica de desempeño se
calcula únicamente dentro de train.

**Salidas.** `reports/data_dictionary.csv`, `reports/figures/audit_*.png`,
`features/dup_clusters.parquet`, `experiments/audit_01_resumen.json`.

## 0. Configuración y entorno

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import audit, config as cfg, cv, data, submission
from src.config import set_seed

set_seed()
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)
print("ROOT:", ROOT)
print("python:", sys.version.split()[0], "| pandas:", pd.__version__, "| numpy:", np.__version__)
print("GPU requerida en esta fase: NO (pipeline CPU-first)")

ROOT: /home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ
python: 3.14.7 | pandas: 2.3.3 | numpy: 1.26.4
GPU requerida en esta fase: NO (pipeline CPU-first)


## 1. Descarga y verificación de integridad

**OBSERVACIÓN esperada:** tres archivos (`age_train.csv`, `age_test.csv`,
`age_submission.csv`) descargados de la competencia privada con autenticación Bearer.

**MÉTODO:** `src.data.download_raw()` (streaming + SHA-256) y `src.data.verify()`, que
revisa el número de campos por registro con `csv.reader` (detecta separadores embebidos y
filas truncadas), unicidad de `id`, coincidencia de ids test↔submission y conteo de textos
vacíos.

In [2]:
info = data.download_raw()
print("ZIP sha256 :", info["zip"])
print("ZIP bytes  :", f"{int(info['size']):,}", "| cacheado:", info.get("cached"))
print("Contenido  :", info["files"])

checks = data.verify()
print(json.dumps(checks, indent=2, ensure_ascii=False))
print("\nSHA256SUMS.txt:")
print(cfg.SHA256_FILE.read_text(encoding="utf-8"))

assert checks["integrity_train"]["bad_records"] == 0
assert checks["integrity_test"]["bad_records"] == 0
assert checks["integrity_sample"]["bad_records"] == 0
assert checks["test_ids_equal_sample_ids"] is True
print("Integridad estructural verificada: 0 registros malformados en los tres archivos.")

ZIP sha256 : 89a8d0586405e55ebbc4e04d2b4d2c68d7e9c4c7d890f4e6712ab8736ff9438e
ZIP bytes  : 15,403,318 | cacheado: true
Contenido  : age_submission.csv,age_test.csv,age_train.csv


{
  "train_rows": 169336,
  "test_rows": 112892,
  "sample_rows": 112892,
  "train_id_unique": true,
  "test_id_unique": true,
  "test_ids_equal_sample_ids": true,
  "train_classes": {
    "13-17": 28863,
    "18-24": 28676,
    "25-34": 28347,
    "35-49": 28230,
    "50-64": 28060,
    "65-xx": 27160
  },
  "train_empty_text": 104,
  "test_empty_text": 62,
  "train_max_text_len": 140,
  "test_max_text_len": 140,
  "integrity_train": {
    "records": 169337,
    "bad_records": 0,
    "examples": []
  },
  "integrity_test": {
    "records": 112893,
    "bad_records": 0,
    "examples": []
  },
  "integrity_sample": {
    "records": 112893,
    "bad_records": 0,
    "examples": []
  }
}

SHA256SUMS.txt:
89a8d0586405e55ebbc4e04d2b4d2c68d7e9c4c7d890f4e6712ab8736ff9438e  tweet-age-2026-puj.zip

Integridad estructural verificada: 0 registros malformados en los tres archivos.


## 2. Esquema y diccionario de datos

**INTERPRETACIÓN.** El esquema es mínimo: `id` (identificador), `text` (única señal
disponible) y `age_range` (objetivo, 6 clases). No hay metadatos de usuario ni temporales,
lo que descarta por completo la línea de features de perfil y concentra el modelado en
representación del texto y estilo.

In [3]:
train = data.load_train()
test = data.load_test()
sample = data.load_sample_submission()

for name, frame in (("train", train), ("test", test), ("submission", sample)):
    print("=" * 78)
    print(json.dumps(audit.basic_profile(frame, name), indent=2, ensure_ascii=False)[:900])

dictionary = audit.build_data_dictionary(train, test, sample)
dictionary.to_csv(cfg.REPORTS / "data_dictionary.csv", index=False)
print("\nreports/data_dictionary.csv escrito:", len(dictionary), "filas")
dictionary

{
  "file": "train",
  "rows": 169336,
  "columns": [
    "id",
    "text",
    "age_range"
  ],
  "dtypes": {
    "id": "int64",
    "text": "object",
    "age_range": "category"
  },
  "missing": {
    "id": 0,
    "text": 0,
    "age_range": 0
  },
  "unique": {
    "id": 169336,
    "text": 168091,
    "age_range": 6
  },
  "id_min": 1,
  "id_max": 169336,
  "id_duplicated": 0
}
{
  "file": "test",
  "rows": 112892,
  "columns": [
    "id",
    "text"
  ],
  "dtypes": {
    "id": "int64",
    "text": "object"
  },
  "missing": {
    "id": 0,
    "text": 0
  },
  "unique": {
    "id": 112892,
    "text": 112245
  },
  "id_min": 1,
  "id_max": 112892,
  "id_duplicated": 0
}
{
  "file": "submission",
  "rows": 112892,
  "columns": [
    "id",
    "age_range"
  ],
  "dtypes": {
    "id": "int64",
    "age_range": "object"
  },
  "missing": {
    "id": 0,
    "age_range": 0
  },
  "unique": {
    "id": 112892,
    "age_range": 1
  },
  "id_min": 1,
  "id_max": 112892,
  "id_duplicated":

,column,type,missing,unique,description,role
0,age_train.id,int64,0,169336,Identificador único del tweet dentro del archi...,id
1,age_train.text,object,0,168091,Contenido del tweet (ancho fijo 140 con paddin...,feature
2,age_train.age_range,category,0,6,Variable objetivo: rango de edad del autor del...,target
3,age_test.id,int64,0,112892,Identificador único del tweet dentro del archi...,id
4,age_test.text,object,0,112245,Contenido del tweet (ancho fijo 140 con paddin...,feature
5,age_submission.id,int64,0,112892,Identificador del tweet a predecir (mismo orde...,id
6,age_submission.age_range,object,0,1,Etiqueta predicha; plantilla con el valor 13-17,prediction


## 3. Perfil de texto: padding de ancho fijo (D-005)

**OBSERVACIÓN.** El corpus almacena el texto en un campo de ancho fijo: la mayoría de las
filas miden exactamente 140 caracteres en bruto, pero la longitud real (tras eliminar el
relleno de espacios) es bastante menor.

**INTERPRETACIÓN.** El relleno es un artefacto de exportación (límite histórico de Twitter),
no una señal autoral; el truncado a 140 caracteres sí es una limitación real del corpus.

**CONSECUENCIA.** `src.data.clean_text` aplica `strip`; el truncado se registra como
hipótesis de error analysis (los tweets largos pierden contexto).

In [4]:
raw_train = (pd.read_csv(cfg.TRAIN_FILE, sep=";", encoding="utf-8-sig", usecols=["text"])["text"]
             .fillna("").astype(str))
raw_test = (pd.read_csv(cfg.TEST_FILE, sep=";", encoding="utf-8-sig", usecols=["text"])["text"]
            .fillna("").astype(str))

lengths = pd.DataFrame([audit.length_profile(train, "train"), audit.length_profile(test, "test")])
lengths.to_csv(cfg.TABLES / "audit_longitudes.csv", index=False)
print(audit.fig_padding_effect(raw_train.str.len(), train["text"].str.len()))
lengths.T

/home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ/reports/figures/audit_padding.png


,0,1
file,train,test
raw_len_min,0,0
raw_len_median,93.0,93.0
raw_len_max,140,140
raw_len_eq_padded_width_pct,2.487,2.537
clean_len_min,0,0
clean_len_median,93.0,93.0
clean_len_mean,90.11,90.15
clean_len_max,140,140
clean_len_eq_padded_width,4211,2864


## 4. Distribución de clases

**OBSERVACIÓN.** Las seis clases están casi perfectamente balanceadas (16,0%–17,0%).

**INTERPRETACIÓN.** El corpus fue construido balanceado por diseño, típico de los esquemas
de *author profiling* (rangos tipo PAN). Un clasificador trivial que prediga la clase
mayoritaria obtendría ≈17% de exactitud.

**CONSECUENCIA.** La métrica macro-F1 y la exactitud serán cercanas entre sí; no se
necesitan técnicas de rebalanceo, y cualquier mejora debe venir de la representación.

In [5]:
counts = train[cfg.TARGET].value_counts().reindex(cfg.CLASSES)
summary = pd.DataFrame({"tweets": counts, "pct": (100 * counts / len(train)).round(3)})
print(audit.fig_class_distribution(train))
print("Clase mayoritaria:", counts.idxmax(), f"({100*counts.max()/len(train):.2f}%)")
print("Desbalanceo máximo relativo: %.3f" % (counts.max() / counts.min()))
summary

/home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ/reports/figures/audit_distribucion_clases.png
Clase mayoritaria: 13-17 (17.04%)
Desbalanceo máximo relativo: 1.063


,tweets,pct
age_range,,
13-17,28863,17.045
18-24,28676,16.934
25-34,28347,16.740
35-49,28230,16.671
50-64,28060,16.571
65-xx,27160,16.039


## 5. Verificación de leakage por `id`

**OBSERVACIÓN.** Si los identificadores se hubieran asignado tras ordenar por clase, `id`
sería un proxy del objetivo y bastaría con explotarlo.

**MÉTODO.** Probabilidad de cada clase por decil de `id` (test de estabilidad).

**RESULTADO.** La desviación máxima respecto a la línea base (1/6) es de ~1 punto
porcentual, sin tendencia monótona.

**DECISIÓN.** Se descarta `id` como feature. Ningún modelo lo usará.

In [6]:
id_table = audit.id_proxy_check(train, n_bins=10)
deviation = (id_table - 1 / len(cfg.CLASSES)).abs().max()
print(audit.fig_id_proxy(id_table))
print("Desviación máxima por clase (pp):", (deviation * 100).round(2).to_dict())
print("Desviación global máxima (pp): %.2f" % (deviation.max() * 100))
id_table

/home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ/reports/figures/audit_id_proxy.png
Desviación máxima por clase (pp): {'13-17': 0.93, '18-24': 0.48, '25-34': 0.54, '35-49': 0.66, '50-64': 0.54, '65-xx': 1.25}
Desviación global máxima (pp): 1.25


age_range,13-17,18-24,25-34,35-49,50-64,65-xx
decile,,,,,,
0,0.1711,0.1678,0.1721,0.1610,0.1683,0.1597
1,0.1663,0.1715,0.1679,0.1630,0.1682,0.1630
2,0.1729,0.1690,0.1642,0.1641,0.1656,0.1642
3,0.1717,0.1701,0.1639,0.1700,0.1648,0.1596
4,0.1658,0.1683,0.1676,0.1713,0.1680,0.1591
5,0.1760,0.1702,0.1705,0.1648,0.1643,0.1542
6,0.1719,0.1684,0.1632,0.1689,0.1654,0.1622
7,0.1741,0.1670,0.1702,0.1657,0.1613,0.1616
8,0.1677,0.1713,0.1655,0.1733,0.1628,0.1593


## 6. Ruido de etiqueta

**OBSERVACIÓN.** Hay textos idénticos (normalizados) anotados con etiquetas distintas.

**INTERPRETACIÓN.** Ruido irreducible: fija un techo práctico de desempeño y explica parte
de los errores que ningún modelo podrá corregir. Se conserva (eliminar filas sesgaría la
distribución), pero se documenta.

In [7]:
conflicts = audit.label_conflict_stats(train)
print(json.dumps(conflicts, indent=2, ensure_ascii=False))

{
  "texts_with_conflict": 203,
  "rows_affected": 643,
  "rows_affected_pct": 0.3797
}


## 7. Duplicados y near-duplicados

**OBSERVACIÓN.** En Twitter abundan los retuits y las variantes del mismo texto (mismo
cuerpo con distinto enlace o menciones). Si un texto de test aparece en train, el modelo
puede memorizarlo en lugar de aprender estilo.

**MÉTODO.** Normalización (minúsculas, sin URLs, sin menciones, sin prefijo `RT`, sin
puntuación) y agrupamiento por texto normalizado idéntico **o** por similitud de **Jaccard
≥ 0,8** entre conjuntos de tokens, con *blocking* por los tokens más raros para evitar el
costo cuadrático. Se agrupa sobre train+test concatenados, sin usar etiquetas.

**NOTA METODOLÓGICA (corrección documentada).** La primera versión agrupaba por la firma de
los primeros 60 caracteres y produjo clústeres espurios: un mismo clúster llegó a reunir 190
tweets con las 6 etiquetas mezcladas, unidos solo por compartir una apertura genérica. Eso
hundía la pureza medida (67%) y hacía ininterpretable la cuantificación del leakage. El
método actual solo une textos realmente similares (comparaciones: ~4,8 millones, presupuesto
no agotado).

**RESULTADO.** Ver tabla y figura: ~1,2% de las filas de test son duplicado exacto de train y
~4,3% comparten clúster de near-duplicado, con ~93% de clústeres de etiqueta única.

**CONSECUENCIA.** Validación agrupada por clúster (D-003) y cuantificación explícita del
puntaje "memorizable" en la sección 8.

In [8]:
exact = audit.duplicate_stats(train, test)
print(json.dumps(exact, indent=2, ensure_ascii=False))

all_texts = pd.concat([train[cfg.TEXT_COL], test[cfg.TEXT_COL]], ignore_index=True)
cluster_stats: dict = {}
clusters = cv.build_dup_clusters(all_texts, stats=cluster_stats)
clusters.to_parquet(cfg.FEATURES / "dup_clusters.parquet", index=False)
print("Diagnóstico de agrupamiento:", json.dumps(cluster_stats, ensure_ascii=False))

leakage = cv.cluster_leakage_report(clusters, n_train=len(train), y_train=train[cfg.TARGET])
print(json.dumps(leakage, indent=2, ensure_ascii=False))
print(audit.fig_dup_exposure(exact["test_rows_seen_in_train_pct"],
                             leakage["test_coverage_pct"],
                             leakage.get("purity_pct", float("nan"))))

sizes = clusters["cluster"].value_counts()
print("Clústeres:", sizes.size, "| tamaño máximo:", int(sizes.max()),
      "| %% train en clústeres de tamaño>1: %.2f" %  # noqa: UP031
      (100 * clusters["cluster"].iloc[:len(train)].duplicated(keep=False).mean()))

train_cluster_ids = clusters["cluster"].to_numpy()[:len(train)]
purity_frame = pd.DataFrame({"cluster": train_cluster_ids, "y": train[cfg.TARGET].astype(str)})
mode_pred = purity_frame.groupby("cluster")["y"].transform(lambda values: values.mode().iat[0])
print("Pureza a nivel de fila (train, moda del clúster): %.2f%%" % (100 * mode_pred.eq(purity_frame["y"]).mean()))
audit.most_common_labels_per_cluster(clusters, n_train=len(train), y_train=train[cfg.TARGET])

{
  "train_exact_dup_rows": 1356,
  "train_exact_dup_pct": 0.8008,
  "train_unique_texts": 167980,
  "test_unique_texts": 112174,
  "test_rows_seen_in_train": 1364,
  "test_rows_seen_in_train_pct": 1.2082,
  "test_unique_texts_seen_in_train": 928
}


Diagnóstico de agrupamiento: {"blocks": 149847, "comparisons": 4795856, "pair_budget_exhausted": false, "threshold": 0.8, "max_df_for_key": 200, "block_tokens": 3}
{
  "n_clusters_total": 271539,
  "n_clusters_train": 164018,
  "n_clusters_test": 109990,
  "n_clusters_crossing": 2469,
  "test_rows_covered": 4804,
  "test_coverage_pct": 4.2554,
  "covered_clusters_pure": 2291,
  "covered_clusters": 2469,
  "purity_pct": 92.791
}
/home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ/reports/figures/audit_leakage_duplicados.png


Clústeres: 271539 | tamaño máximo: 265 | % train en clústeres de tamaño>1: 4.37


Pureza a nivel de fila (train, moda del clúster): 99.37%


,cluster,size_train,labels,example
0,141,153,"{'13-17': 34, '25-34': 29, '18-24': 14, '35-49...",acabo de publicar una foto
1,637,131,"{'50-64': 66, '25-34': 10, '18-24': 14, '35-49...",publique una nueva foto en facebook
2,2104,78,"{'18-24': 48, '13-17': 23, '25-34': 6, '35-49'...",seu perfil foi visto por 7 pessoas nas ultimas...
3,1967,72,"{'18-24': 69, '25-34': 2, '50-64': 1}",gracias por seguirme en breve te devuelvo foll...
4,5559,61,{'35-49': 61},colombia time esta disponible
5,1670,58,"{'13-17': 43, '18-24': 13, '35-49': 1, '25-34'...",25 usuarios que sigo no me siguen de vuelta en...
6,4387,56,{'35-49': 56},el socialmedia diario is out stories via
7,5555,55,{'35-49': 55},el techno diario is out stories via
8,1326,54,{'50-64': 54},25 s 25 personas le han dado like a tu tuit
9,8049,54,{'35-49': 54},jnoelsalcedo videogame post is out stories via


## 8. ¿Cuánto puntaje es memorizable? (cuantificación del leakage)

**PREGUNTA.** ¿Cuánto del desempeño esperado puede obtenerse memorizando duplicados en
lugar de generalizar estilo?

**MÉTODO.** Dentro de train, con validación cruzada: para cada fila de validación cuyo
clúster tiene miembros en el train del fold, se predice la etiqueta mayoritaria del clúster;
el resto recibe un centinela que nunca acierta. Se compara el esquema **agrupado por
clúster** (primario) contra el **estratificado ingenuo** (control).

**INTERPRETACIÓN.** La diferencia entre ambos esquemas mide el optimismo del split aleatorio
y justifica la validación agrupada en todas las fases siguientes.

In [9]:
cluster_ids = clusters["cluster"].to_numpy()
y_train = train[cfg.TARGET].astype(str).to_numpy()

folds_group = cv.make_folds(y_train, groups=cluster_ids[:len(train)], scheme="group")
folds_strat = cv.make_folds(y_train, groups=None, scheme="stratified")

mem_group = cv.duplicate_memorization_baseline(cluster_ids[:len(train)], y_train, folds_group)
mem_strat = cv.duplicate_memorization_baseline(cluster_ids[:len(train)], y_train, folds_strat)

comparison = pd.DataFrame({
    "esquema": ["agrupado por clúster (primario)", "estratificado ingenuo (control)"],
    "cobertura_%": [mem_group["coverage_pct"], mem_strat["coverage_pct"]],
    "acierto_en_cubiertas": [mem_group["accuracy_covered"], mem_strat["accuracy_covered"]],
    "macro_f1_memorizacion": [mem_group["macro_f1_memorization_only_mean"],
                              mem_strat["macro_f1_memorization_only_mean"]],
    "macro_f1_std": [mem_group["macro_f1_memorization_only_std"],
                     mem_strat["macro_f1_memorization_only_std"]],
})
comparison.to_csv(cfg.TABLES / "audit_memorizacion.csv", index=False)
print(json.dumps({"group": mem_group, "stratified": mem_strat}, indent=2, ensure_ascii=False))
comparison

{
  "group": {
    "coverage_pct": 0.0,
    "accuracy_covered": 0.0,
    "macro_f1_memorization_only_mean": 0.0,
    "macro_f1_memorization_only_std": 0.0
  },
  "stratified": {
    "coverage_pct": 3.9875,
    "accuracy_covered": 0.7952,
    "macro_f1_memorization_only_mean": 0.0608,
    "macro_f1_memorization_only_std": 0.0014
  }
}


,esquema,cobertura_%,acierto_en_cubiertas,macro_f1_memorizacion,macro_f1_std
0,agrupado por clúster (primario),0.0000,0.0000,0.0000,0.0000
1,estratificado ingenuo (control),3.9875,0.7952,0.0608,0.0014


## 9. Validación de la plantilla de submission

**MÉTODO.** `src.submission.validate_submission` comprueba columnas, orden, número de
filas, ids, etiquetas válidas, NaN y duplicados. Se valida la plantilla oficial y una
submission ficticia (todo `13-17`) para demostrar que el validador acepta lo correcto y
rechaza lo incorrecto.

**RESULTADO.** La plantilla oficial pasa todas las comprobaciones; el validador está listo
para usarse en la Fase 22 sin cambios.

In [10]:
official = submission.validate_submission(sample.copy(), test[cfg.ID_COL], sample)
print("plantilla oficial:", official)

dummy = pd.DataFrame({cfg.ID_COL: test[cfg.ID_COL], cfg.TARGET: "13-17"})
print("submission ficticia válida:", submission.validate_submission(dummy, test[cfg.ID_COL], sample))

bad = dummy.copy()
bad.loc[0, cfg.TARGET] = "99-99"
try:
    submission.validate_submission(bad, test[cfg.ID_COL], sample)
    print("ERROR: el validador no detectó la etiqueta inválida")
except submission.SubmissionError as exc:
    print("rechazo correcto de etiqueta inválida:", str(exc)[:120])

plantilla oficial: {'columns_ok': True, 'columns_match_sample': True, 'rows_ok': True, 'ids_equal_test': True, 'ids_unique': True, 'ids_same_order': True, 'no_nan': True, 'labels_valid': True}
submission ficticia válida: {'columns_ok': True, 'columns_match_sample': True, 'rows_ok': True, 'ids_equal_test': True, 'ids_unique': True, 'ids_same_order': True, 'no_nan': True, 'labels_valid': True}
rechazo correcto de etiqueta inválida: Submission inválida: fallan ['labels_valid']. Detalle: {'rows': (112892, 112892), 'labels_encontradas': ['13-17', '99-99


## 10. Gate A — Resumen de la auditoría

Se consolida la evidencia en `experiments/audit_01_resumen.json` y se listan los hechos que
condicionan el resto del proyecto.

In [11]:
resumen = {
    "competition": cfg.COMPETITION,
    "zip_sha256": info["zip"],
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "classes": {str(key): int(value) for key, value in counts.items()},
    "empty_texts": {"train": int((train[cfg.TEXT_COL] == "").sum()),
                    "test": int((test[cfg.TEXT_COL] == "").sum())},
    "padding": lengths.set_index("file").to_dict(orient="index"),
    "id_leakage_max_deviation_pp": round(float(deviation.max() * 100), 3),
    "label_conflicts": conflicts,
    "exact_duplicates": exact,
    "cluster_leakage": leakage,
    "memorization_group": mem_group,
    "memorization_stratified": mem_strat,
    "submission_format_ok": official,
}
(cfg.EXPERIMENTS / "audit_01_resumen.json").write_text(
    json.dumps(resumen, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: resumen[k] for k in
                  ("train_rows", "test_rows", "id_leakage_max_deviation_pp", "label_conflicts",
                   "memorization_group", "memorization_stratified")}, indent=2, ensure_ascii=False))
print("\nResumen guardado en experiments/audit_01_resumen.json")

{
  "train_rows": 169336,
  "test_rows": 112892,
  "id_leakage_max_deviation_pp": 1.247,
  "label_conflicts": {
    "texts_with_conflict": 203,
    "rows_affected": 643,
    "rows_affected_pct": 0.3797
  },
  "memorization_group": {
    "coverage_pct": 0.0,
    "accuracy_covered": 0.0,
    "macro_f1_memorization_only_mean": 0.0,
    "macro_f1_memorization_only_std": 0.0
  },
  "memorization_stratified": {
    "coverage_pct": 3.9875,
    "accuracy_covered": 0.7952,
    "macro_f1_memorization_only_mean": 0.0608,
    "macro_f1_memorization_only_std": 0.0014
  }
}

Resumen guardado en experiments/audit_01_resumen.json


### Conclusiones del gate A

| # | Observación | Interpretación | Hipótesis | Consecuencia |
|---|---|---|---|---|
| A1 | 6 clases balanceadas (16,0–17,0%) | corpus balanceado por diseño (esquema tipo PAN) | el desbalanceo no es un problema; la señal está en el estilo | sin rebalanceo; macro-F1 como métrica primaria |
| A2 | Texto truncado/padded a 140 chars; mediana real 93 | límite histórico de Twitter, artefacto de exportación | los tweets largos truncados pierden señal y serán más difíciles | `strip` del padding; hipótesis de error analysis por longitud |
| A3 | `id` no es proxy de clase (≤1 pp) | asignación aleatoria de ids | — | `id` descartado como feature |
| A4 | 1,2% de test aparece literal en train; 4,3% comparte clúster de near-duplicado (Jaccard ≥ 0,8), con 93% de clústeres puros | leakage por duplicados (retuits y variantes) | memorizar duplicados infla el score si el CV no los agrupa | CV agrupado por clúster (D-003) y cuantificación del efecto |
| A5 | 0,38% de filas con etiqueta contradictoria | ruido irreducible del corpus | techo práctico de desempeño | se conserva y se documenta |
| A6 | 166 textos vacíos (104 train, 62 test) | tweets sin contenido (solo enlace o vacío) | sin señal textual: predicción por prior | manejo explícito en el pipeline |
| A7 | Sin metadatos de usuario | no hay author profiling multiusuario | la señal es 100% textual | foco en representación léxica, estilométrica y de carácter |

**Próxima fase:** Fase 1 (estado del arte en dos corrientes) y Fase 2-3 (EDA y análisis
lingüístico) con la lista priorizada de hipótesis.